# Классические модели. Работа с широкими данными

автор: Алексей Космачев

В общем случае данные для машинного обучения могут быть большими в двух смыслах

* Очень много наблюдений. Визуализируя данные для обучения как таблицу наблюдения-признаки, можно это представлять как очень "высокую" таблицу с огромных количеством записей. По этой причине такую ситуацию еще называют "высокими данными".

* Очень много признаков. Проводя такую же аналогию с таблицей, она бы выглядела как очень "широкая таблица" с гигантским количеством столбцов. Такую ситуацию аналогично называют "широкими данными".

Обе ситуации требуют различных подходов для решения. В этом семинаре будет говорить про "широкие данные".

### А где встречаются широкие данные?

Ярким примером являются рекомндательные системы. Там любой user_id/item_id это признак, а их может быть бесконечно много. По этому поводу есть интересная работа - [SLIM](https://www-users.cse.umn.edu/~ningx005/slides/ICDM2011_slides.pdf) - интерпретируемая линейная регрессия для рекомендательных систем. Об этом можно почитать, например, здесь: [телеграм](https://t.me/WazowskiRecommends/14) или [медиум](https://roizner.medium.com/slim-a-fast-and-interpretable-baseline-for-recommender-algorithms-559cd73dcc55) (тот же пост на английском)

### Подготовим данные и инструменты для дальшей работы

In [1]:
! mkdir -p sem06
%cd sem06

/home/ubuntu/sem06


Для демонстрационных целей будем использовать датасет с отзывами пользователей на различные медицинские препараты

Больше информации на UCI - https://archive.ics.uci.edu/ml/datasets/Drug+Review+Dataset+%28Drugs.com%29

In [2]:
! wget https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
! unzip drugsCom_raw.zip

--2026-02-24 14:20:48--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip’

drugsCom_raw.zip        [           <=>      ]  41.00M   440KB/s    in 2m 40s  

2026-02-24 14:23:29 (262 KB/s) - ‘drugsCom_raw.zip’ saved [42989872]

/bin/bash: unzip: command not found


In [4]:
! unzip drugsCom_raw.zip

Archive:  drugsCom_raw.zip
  inflating: drugsComTest_raw.tsv    
  inflating: drugsComTrain_raw.tsv   


In [5]:
! head drugsComTest_raw.tsv

Основной инструмент, который будем сегодня изучать - это Vowpal Wabbit.

Больше информации можно найти на официальном сайте - https://vowpalwabbit.org/

Ниже скачаем уже собранную версию VW , готовую к использованию

In [6]:
! whoami

ubuntu


In [7]:
! sudo wget http://finance.yendor.com/ML/VW/Binaries/vw-8.20190624 -O /usr/bin/vw
! sudo chmod +x /usr/bin/vw
! sudo chown ubuntu /usr/bin/vw

--2026-02-24 14:25:24--  http://finance.yendor.com/ML/VW/Binaries/vw-8.20190624
Resolving finance.yendor.com (finance.yendor.com)... 75.119.201.70
Connecting to finance.yendor.com (finance.yendor.com)|75.119.201.70|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://finance.yendor.com/ML/VW/Binaries/vw-8.20190624 [following]
--2026-02-24 14:25:24--  https://finance.yendor.com/ML/VW/Binaries/vw-8.20190624
Connecting to finance.yendor.com (finance.yendor.com)|75.119.201.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9467376 (9.0M)
Saving to: ‘/usr/bin/vw’

/usr/bin/vw         100%[===================>]   9.03M  4.85MB/s    in 1.9s    

2026-02-24 14:25:27 (4.85 MB/s) - ‘/usr/bin/vw’ saved [9467376/9467376]



И заодно поставим всякие стандартные питоновские библиотеки для анализа данных

In [9]:
! sudo apt-get update -y && sudo apt-get install graphviz -y

Hit:1 http://mirror.yandex.ru/ubuntu focal InRelease
Hit:2 http://mirror.yandex.ru/ubuntu focal-updates InRelease                   
Hit:3 http://mirror.yandex.ru/ubuntu focal-backports InRelease                 
Hit:4 http://dataproc.storage.yandexcloud.net/ci/trunk/225-54615002560eee21 focal InRelease
Hit:5 http://security.ubuntu.com/ubuntu focal-security InRelease               
Get:6 https://repos.influxdata.com/ubuntu focal InRelease [12.9 kB]            
Err:7 http://mirror.yandex.ru/mirrors/postgresql focal-pgdg InRelease          
  421  Misdirected Request [IP: 213.180.204.183 80]
Err:6 https://repos.influxdata.com/ubuntu focal InRelease                      
  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY DA61C26A0585BD3B
Hit:8 https://packages.fluentbit.io/ubuntu/focal focal InRelease            
Hit:9 https://packages.broadcom.com/artifactory/saltproject-deb stable InRelease
Reading package lists... Done
E: Failed to fetch 

In [10]:
! pip install numpy pandas scikit-learn dateparser pandarallel ipywidgets catboost graphviz

     |████████████████████████████████| 17.3 MB 1.6 MB/s eta 0:00:01
     |████████████████████████████████| 12.4 MB 49.2 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 53.4 MB/s eta 0:00:01
     |████████████████████████████████| 315 kB 80.9 MB/s eta 0:00:01
     |████████████████████████████████| 139 kB 74.0 MB/s eta 0:00:01
     |████████████████████████████████| 97.1 MB 115 kB/s  eta 0:00:01
     |████████████████████████████████| 47 kB 7.9 MB/s  eta 0:00:01
     |████████████████████████████████| 348 kB 61.1 MB/s eta 0:00:01
     |████████████████████████████████| 229 kB 59.8 MB/s eta 0:00:01
     |████████████████████████████████| 509 kB 70.9 MB/s eta 0:00:01
     |████████████████████████████████| 301 kB 60.5 MB/s eta 0:00:01
     |████████████████████████████████| 34.5 MB 57.1 MB/s eta 0:00:01
     |████████████████████████████████| 785 kB 73.9 MB/s eta 0:00:01
     |████████████████████████████████| 119 kB 76.9 MB/s eta 0:00:01
     |█████████████████████████

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [11]:
!pip install --upgrade --force-reinstall pandarallel

Processing /home/ubuntu/.cache/pip/wheels/c7/e9/78/48eb140b79de41c4d9440938ef5f9e3186c979183fe57829b3/pandarallel-1.6.5-py3-none-any.whl
  Using cached psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl (155 kB)
  Using cached pandas-2.0.3-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.4 MB)
  Using cached dill-0.4.0-py3-none-any.whl (119 kB)
  Using cached numpy-1.24.4-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.3 MB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
ERROR: launchpadlib 1.10.13 requires testresources, which is not installed.
  Attempting uninstall: psutil
    Found existing installation: psutil 7.2.2
    Uninstalling psutil-7.2.2:
      Successfully uninstalled psutil-7.2.2
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.4
    Uninstal

In [12]:
! pip install psutil

In [13]:
! /opt/conda/bin/jupyter nbextension enable --py widgetsnbextension

Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [15]:
# Проверяем, что vw работает
! vw --help | head

Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = 
num sources = 1
driver:
  --onethread           Disable parse thread
VW options:
  --ring_size arg (=256, ) size of example ring
  --strict_parse           throw on malformed examples
Update options:
  -l [ --learning_rate ] arg Set learning rate
  --power_t arg              t power value
  --decay_learning_rate arg  Set Decay factor for learning_rate between passes
  --initial_t arg            initial t value


## Хеширование признаков. Линейные модели.

Vowpal Wabbit - инструмент от компании Microsoft, который позволяет очень эффективно обучать линейные модели. Ключевая особенность данного инструмента - использования хешинг-трика. То есть все признаки изначально хешируются, что позволяет модели эффективно обрабатывать практически произвольное количество признаков.

In [12]:
изначальные признаки:
x1 = 1
x2 = 0
.
.
.
xn = 5

n >> 1M

что хотим получить по набору признаков:
y1
y2
.
.
.
yk

k << n


h - hash function, h : x_{i} -> [0, k)
h(x_{i}) \forall i

h(x1)
h(x2)
...
h(xn)
                                 
                                 
h(xi) = h(xj) 
value(xi), value(xj) -> mean(value(xi), value(xj)), max(value(xi), value(xj)), min(value(xi), value(xj))


[1, 0, 0, 0, ....., 1, 0, ..., 1]
[0, 0, 0, 0, ....., 0, ...1, ...0, ..., 1]
x1, ..., xm, m << n (n - maximum)
hash(xm+t) = h(xm)


SyntaxError: invalid syntax (<ipython-input-12-05c193904dc7>, line 1)

### Формат данных
VW для обучения требует свой особенный формат хранения данных. Он выглядит следующим образом:

```
[Label] [Importance] [Tag]|Namespace Features |Namespace Features ... |Namespace Features
```

* Label - значение целевой переменной. Если не указывать, объект не будет использоваться в обучении
* Importance - вес объекта. Если не указывать, равен 1
* Tag - пометка объекта. Никак не влияет на процесс обучения, но добавляет семантики и "читаемости" данных для человека
* Namespace - название области признаков. Используется, чтобы разные по сути признаки с одинаковым названием не пересекались
* Features - признаки. Это или пара <название признака>:<значение признака> или просто <название признака>. В последнем случае будет считаться, что значение равно 1

Например, попробуем закодировать несколько элементов из Iris в этом формате

| Длина чашелистика | Ширина чашелистика | Длина лепестка | Ширина лепестка |  Вид ириса |
|:-----------------:|:------------------:|:--------------:|:---------------:|:----------:|
| 5.1               | 3.5                | 1.4            | 0.2             | setosa     |
| 4.9               | 3.0                | 1.4            | 0.2             | setosa     |
| 7.0               | 3.2                | 4.7            | 1.4             | versicolor |

Будем считать setosa классом 1, а versicolor классом -1.

In [16]:
%%writefile iris_1.example.vw
1 | sepal_length:5.1 sepal_width:3.5 petal_length:1.4 petal_width:0.2
1 | sepal_length:4.9 sepal_width:3.0 petal_length:1.4 petal_width:0.2
-1 | sepal_length:7.0 sepal_width:3.2 petal_length:4.7 petal_width:1.4

Writing iris_1.example.vw


В данном наборе мы дополнительно можем сгруппировать признаки - признаки для чашелистика и признаки для лепестка. Это позволит логически их разделить, сэкономит место, а также позволит в дальнейшем оперировать этими группами.

Запишем точно эти же данные, но с использованием именованных секций.

In [17]:
%%writefile iris_2.example.vw
1 |sepal length:5.1 width:3.5 |petal length:1.4 width:0.2
1 |sepal length:4.9 width:3.0 |petal length:1.4 width:0.2
-1 |sepal length:7.0 width:3.2 |petal length:4.7 width:1.4

Writing iris_2.example.vw


Подобный формат записи данных позволяет очень удобно записывать категориальные признаки в формате One-Hot Encoding. При таком кодировании мы бы для признака, соответствующего конкретной категории, выставили бы значение 1, а всем остальным - 0.

Например для признака с полом человека превратится в два признака

| gender_man | gender_woman |
|:----------:|:------------:|
| 1          | 0            |

Если же в VW не указывать явно значение признака, то оно будет в автоматическом режиме выставлено равным 1, чего мы и ожидаем при One-Hot кодировании.

Попробуем для примера записать вот такие данные в этом формате

| Класс |  Цвет фона | Включена ли темная тема | Размер шрифта | Межстрочный интервал |
|:-----:|:----------:|:-----------------------:|:-------------:|:--------------------:|
| 1     | White      | да                      | 12            | 1.5                  |
| -1    | Black      | нет                     | 14            | 1.5                  |
| 1     | White, Red | нет                     | 12            | 2                    |
| -1    | Black, Red | да                      | 15            | 1.5                  |

In [19]:
%%writefile design.example.vw
1 |color white |theme dark |font size:12.0 interval:1.5
-1 |color black |theme normal |font size:14.0 interval:1.5
1 |color white red |theme normal:1 |font size: 12.0 interval:2.0
-1 |color black red |theme dark |font size: 15.0 interval:1.5

Overwriting design.example.vw


Можно заметить, что такой подход к кодированию также позволяет очень просто кодировать текстовые признаки в формате bag-of-words. При использовании bag-of-words мы для каждого слова добавляет отдельный бинарный признак, который равен 1, если слово есть в тексте.

Таким образом, чтобы закодировать в формате vw какой-то текст через bag-of-words достаточно... Просто написать этот текст! Единственное, что необходимо сделать дополнительно - это очистить сам текст от лишних символов, таких как знаки препинания, удаления, кавычки и так далее. Очищенный текст - это и есть кодирование признаков в vw.

Попробуем, например, закодировать вот такой набор данных.

| Оценка |                             Заголовок                              |                                                                      Комментарий                                                                     |
|:------:|:------------------------------------------------------------------:|:----------------------------------------------------------------------------------------------------------------------------------------------------:|
| 2      | To lazy to watch film                                              | My family and I normally do not watch local movies for the simple reason that they are poorly made                                                   |
| 5      | Well-directed and fairly notorious piece of Italian nunsploitation | This file is a genuinely moving and intelligent movie with plenty of nudity and gore.You can't go wrong with it.5 out of 5.                         |
| 3      | this was at one time the worst movie I had ever seen               | But since than time, I have seen many more movies that are worse (how is it possible??) Therefore, to be fair, I had to give this movie a 3 out of 5 |

In [20]:
%%writefile reviews.vw
2.0 |title to lazy to watch film |review my family and i normally do not watch local movies for the simple reason that they are poorly made
5.0 |title well directed and fairly notorious piece of italian nunsploitation |review this file is a genuinely moving and intelligent movie with plenty of nudity and gore you can t go wrong with it 8 out of 10 
3.0 |title this was at one time the worst movie i had ever seen |review but since than time  i have seen many more movies that are worse how is it possible  therefore to be fair  i had to give this movie a 3 out of 5 

Writing reviews.vw


In [21]:
! cat reviews.vw

2.0 |title to lazy to watch film |review my family and i normally do not watch local movies for the simple reason that they are poorly made
5.0 |title well directed and fairly notorious piece of italian nunsploitation |review this file is a genuinely moving and intelligent movie with plenty of nudity and gore you can t go wrong with it 8 out of 10 
3.0 |title this was at one time the worst movie i had ever seen |review but since than time  i have seen many more movies that are worse how is it possible  therefore to be fair  i had to give this movie a 3 out of 5 


В этом примере именованные секции играют важную роль, так как одно и тоже слово может оказаться и в заголовке и в тексте комментария. Без секций мы бы считали это одним признаком. Секции позволяют нам разделять слова, которые использовались в заголовке от слов, которые использовались в описании.

Итак, мы научились формировать данные для VW, самое время запустить его. Попробуем использовать наши примеры.

In [22]:
! cat reviews.vw

2.0 |title to lazy to watch film |review my family and i normally do not watch local movies for the simple reason that they are poorly made
5.0 |title well directed and fairly notorious piece of italian nunsploitation |review this file is a genuinely moving and intelligent movie with plenty of nudity and gore you can t go wrong with it 8 out of 10 
3.0 |title this was at one time the worst movie i had ever seen |review but since than time  i have seen many more movies that are worse how is it possible  therefore to be fair  i had to give this movie a 3 out of 5 


In [23]:
## запуск vw
! vw reviews.vw --final_regressor reviews.model.bin

final_regressor = reviews.model.bin
Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = reviews.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
4.000000 4.000000            1            1.0   2.0000   0.0000       25
13.653724 23.307447            2            2.0   5.0000   0.1722       36

finished run
number of examples = 3
weighted example sum = 3.000000
weighted label sum = 10.000000
average loss = 10.574909
best constant = 3.333333
total feature number = 106


In [24]:
! cat reviews.model.bin

   8.7.0     m      �@                      �dA�    ;9�=�"  ��=�.  ;9�=�6  �#k=Q=  �#k=QN  ��=>d   a0=�m  DC�=Fx  ��=��   a0=ݕ   a0=�  �#k=H�  ;9�=:�   a0=G�  �p�=��   a0=��  ��=�  ��=9�   a0=� �#k=�2 ��=4 ��=@ H
�  a0=� ��=� ��=� �p�=k� ;9�=\� 4
>p� �#k=L�  a0=Z�  a0=� �#k=��  a0==�  a0=�� �#k=�! �#k=*7 ��=�8 �ly=�:  a0=�; ��=�Q �#k=�U �#k=KY ��=�] �#k=_ �>ri ��=?�  a0=N� �#k=�� ��=К  a0=�  a0=�� ��=F� �#k=�  a0=B�  a0=�� ��=��  a0=� �#k=k  a0=� ��=� SQ�=y.  a0=MB  a0=+Q  a0=\T ��=�Y ��=�o ;9�=o ;9�=$� �#k=�  a0=v� �#k=�� �p�=]� �a�=�� �#k=�� ��=+�  a0=�� ��=��  a0=

На экране можно видеть отчет обучения, а в файл `reviews_result.vw.bin` записались веса модели. По-умолчанию используется линейная регрессия с MSE.

Для того, чтобы получить предсказания нет отдельной команды. Это связано с особым форматом работы инструмента. Можно считать, что vw всегда запускается в режиме обучения, однако есть несколько флагов, которые умеют контролировать этот процесс.

Схематично это можно изобразить на картинке

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vw-explanation.png" width="600">

Таким образом при каждом запуске VW получает входные данные, с которыми он сейчас будет работать. Далее он начинает обучение на этих данных. Если указан `--initial_regressor` то он возьмет указанные веса в качестве начальных - то есть попытается "дообучить" модель, которую мы указали. Параллельно с этим для каждого пройденного элемента он производит предсказание. Если указать параметр `--predictions`, то эти данные он отдельно запишет в файл. По окончанию обучения новые полученные веса он попробует сохранить в файл, указанный через `--final_regressor`. И также есть очень важный флаг `--testonly`, говорящий о том, что обновлять веса (то есть обучаться) в процессе не требуется.

Подобная схема позволяет более гибко задавать режимы работы. Вот возможные сценарии работы, которые можно выставить этими ключами.

* `--final_regressor` - просто обучаем модель на данных
* `--testonly`, `--initial_regressor`, `--predictions` - просто делаем предсказания модели на данных
* `--final_regressor`, `--initial_regressor` - дообучаем модель на новых данных

И так далее. Попробуем сделать предсказание на нашем же примере.

In [25]:
! vw --testonly \
    --initial_regressor reviews.model.bin \
    --predictions reviews_prediction.txt \
    reviews.vw

only testing
predictions = reviews_prediction.txt
Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = reviews.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
0.042279 0.042279            1            1.0   2.0000   1.7944       25
2.310258 4.578236            2            2.0   5.0000   2.8603       36

finished run
number of examples = 3
weighted example sum = 3.000000
weighted label sum = 10.000000
average loss = 1.583771
best constant = 3.333333
total feature number = 106


In [26]:
! cat reviews_prediction.txt

1.794381
2.860319
2.638340


In [27]:
! cat reviews.vw

2.0 |title to lazy to watch film |review my family and i normally do not watch local movies for the simple reason that they are poorly made
5.0 |title well directed and fairly notorious piece of italian nunsploitation |review this file is a genuinely moving and intelligent movie with plenty of nudity and gore you can t go wrong with it 8 out of 10 
3.0 |title this was at one time the worst movie i had ever seen |review but since than time  i have seen many more movies that are worse how is it possible  therefore to be fair  i had to give this movie a 3 out of 5 


### Решаем реальную задачу

Подключим все необходимые библиотеки и прочитаем данные

In [28]:
!wget http://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip

--2026-02-24 14:28:40--  http://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip.1’

drugsCom_raw.zip.1      [                <=> ]  41.00M  10.4MB/s    in 4.1s    

2026-02-24 14:28:44 (9.91 MB/s) - ‘drugsCom_raw.zip.1’ saved [42989872]



In [29]:
!unzip drugsCom_raw.zip

Archive:  drugsCom_raw.zip
replace drugsComTest_raw.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [30]:
import pandas as pd
import numpy as np
import re
import dateparser
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.feature_extraction.text import CountVectorizer

In [31]:
!sudo pip install --upgrade psutil

     |████████████████████████████████| 155 kB 761 kB/s eta 0:00:01


In [32]:
from pandarallel import pandarallel
pandarallel.initialize(use_memory_fs=False, progress_bar=True)

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [33]:
df_train = pd.read_csv("drugsComTrain_raw.tsv", delimiter='\t')
df_test = pd.read_csv("drugsComTrain_raw.tsv", delimiter='\t')
df = pd.concat([df_train, df_test])

In [34]:
df.dropna(inplace=True)

In [35]:
df

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9.0,"May 20, 2012",27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9.0,"November 27, 2016",37
...,...,...,...,...,...,...,...
161292,191035,Campral,Alcohol Dependence,"""I wrote my first report in Mid-October of 201...",10.0,"May 31, 2015",125
161293,127085,Metoclopramide,Nausea/Vomiting,"""I was given this in IV before surgey. I immed...",1.0,"November 1, 2011",34
161294,187382,Orencia,Rheumatoid Arthritis,"""Limited improvement after 4 months, developed...",2.0,"March 15, 2014",35
161295,47128,Thyroid desiccated,Underactive Thyroid,"""I&#039;ve been on thyroid medication 49 years...",10.0,"September 19, 2015",79


Задачу для обучения мы можем поставить себе произвольную. Чтобы было наглядно и в обучающей выборке было много разных признаков сформулируем задачу такую - **По написанному отзыву одного пользователя, определить какому количеству других пользователей эта информация окажется полезной**

Таким образом у нас изначально есть всего 4 признака следующей природы
* `drugName` - категория
* `condigion` - категория
* `review` - произвольный текст
* `rating` - вещественное число

А целевая переменная это `usefulCount` - тоже вещественное число

Подсчитаем количество признаков, которое нам потребуется для кодирования всех этих признаков, чтобы их можно было подать в линейную модель.

Категории будем колировать One-Hot, текст через Bag-of-Words, из временной метки сделаем категориальную переменную с днями недели, а вещественные будет подавать как есть.

In [36]:
len(df['drugName'].unique())  # Количество уникальных препаратов

3431

In [37]:
len(df['condition'].unique())  # Количество уникальных заболеваний

884

In [38]:
len(CountVectorizer().fit(df['review']).get_feature_names_out()) # Количество уникальных слов в корпусе


49855

Итого уже 3431 + 884 + 49855 + 1 = **54 178** признаков для кодирования.

In [39]:
def convert_to_vw(data):
    target = data['usefulCount']
    
    drug_name = data['drugName'].lower().replace(' ', '_')
    condition = data['condition'].lower().replace(' ', '_')
    
    raw_text = data['review'].lower()
    word_pattern = re.compile(r"[a-zA-Z0-9_]+")
    words = [match.group(0) for match in re.finditer(word_pattern, raw_text)]
    review = ' '.join(words)
    
    rating = data['rating']

    
    template = "{target} |d {drug_name} |c {condition} |r {review} |s rating:{rating}"
    return template.format(
        target=target,
        drug_name=drug_name,
        condition=condition,
        review=review,
        rating=rating
    )

In [40]:
_, example = next(df.iterrows())
example

Unnamed: 0                                                206461
drugName                                               Valsartan
condition                           Left Ventricular Dysfunction
review         "It has no side effect, I take it in combinati...
rating                                                       9.0
date                                                May 20, 2012
usefulCount                                                   27
Name: 0, dtype: object

In [41]:
convert_to_vw(example)

'27 |d valsartan |c left_ventricular_dysfunction |r it has no side effect i take it in combination of bystolic 5 mg and fish oil |s rating:9.0'

In [42]:
def write_vw(dataframe, filename):
    with open(filename, 'w') as f:
        for vw_record in dataframe.parallel_apply(convert_to_vw, axis=1):
            f.write(vw_record + '\n')

In [43]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=420)

In [44]:
%%time

write_vw(df_train, 'drugs.train.vw')
write_vw(df_test, 'drugs.test.vw')

CPU times: user 841 ms, sys: 872 ms, total: 1.71 s
Wall time: 10.2 s


In [45]:
! head -n 2 drugs.train.vw

17 |d anaprox-ds |c pain |r i recommend if for girls if they have menstrual pain it helps alot 10 15 mins after i take the pill i feel no more pain |s rating:10.0
1 |d levetiracetam |c epilepsy |r as a 42 year old female i have had seizures my entire life i have taken 14 anti convulsants at one point or another in an attempt to control my complex partial seizures but keppra is the worst medication i have ever taken the side effects are horrible although it 039 s the only medication that has been able to stop my seizures the side effects are so terrible that it 039 s not worth taking it i couldn 039 t string together a coherent thought had short term memory loss had awful mood swings and suddenly hated life everything was difficult i couldn 039 t think straight and felt emotionally raw all the time i could barely keep it together while on this medication every day of life was awful while on this medication i will never take it again |s rating:1.0


Итого мы разбили выборку на обучающую и тестовую и закодировали эти данные в нужном формате. Можно начинать обучать машины!

#### Запускаем чистый VW

Поставим обучаться VW без дополнительных параметров на нашем датасете

In [46]:
! cat drugs.train.vw | wc -l

256636


In [47]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw

final_regressor = drugs.model.bin
Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       30
145.348644 1.697287            2            2.0   1.0000   2.3028      145
240.409956 335.471268            4            4.0   7.0000   7.9989      157
1483.276169 2726.142381            8            8.0  24.0000   4.0544       73
827.101424 170.926679           16           16.0  11.0000  14.8396      137
2549.058402 4271.015380           32           32.0 135.0000   6.7423       55
2170.529700 1792.000997           64           64.0   9.0000  32.8455      150
1650.839727 1131.149754          128          128.0   4.0000  32.2044      149
1504.761907 1358.684087          256          256.0   3.0000  14

In [48]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

only testing
predictions = drugs.preductions.txt
Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000       74
1793.657227 3551.314453            2            2.0  76.0000  16.4071       10
1092.279785 390.902344            4            4.0   4.0000   0.0000       68
884.345149 676.410512            8            8.0  23.0000  20.4742       44
507.483603 130.622057           16           16.0   7.0000  23.0774      126
508.988668 510.493733           32           32.0  13.0000  17.7603       47
596.315355 683.642042           64           64.0   3.0000  10.9192      108
1069.053975 1541.792594          128          128.0   9.0000  27.1967      115
999.484723 929.915471          256          256.0  29

In [49]:
! head drugs.preductions.txt

0
16.407093
49.673176
0
25.992844
31.923294
32.541691
20.474159
13.278885
41.239552


Для проверки качества регрессии будем считать обычный `R^2`. Для того, чтобы делать это просто, напишем функцию, которая будет считывать правильные ответы из vw файла, считывать предсказания и для них подсчитывать R2.

In [50]:
def calc_r2(predictions_filename, answers_filename):
    def read_target_from_vw(vw_record):
        return float(vw_record.split(' ')[0])
    
    with open(predictions_filename, 'r') as f:
        y_pred = np.array([float(value) for value in f.readlines()])
        
    with open(answers_filename, 'r') as f:
        y_expected = np.array([read_target_from_vw(value) for value in f.readlines()])
        
    return r2_score(y_expected, y_pred)

In [51]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.18962522751793354

Качество пока не поражает воображение

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_1.png" width="300">

Основная причина - по умолчанию vw запускает всего одну эпоху с дефолтными параметрами. Этого явно недостаточно, чтобы выбить хорошее качество. Попробуем улучить качество, используя различные опции этого инструмента

In [52]:
! ls -lh drugs.train.vw

-rw-r--r-- 1 ubuntu ubuntu 123M Feb 24 14:30 drugs.train.vw


In [53]:
! 

#### Эпохи, размер хеша, лернинг рейт

Начнем тюнить нашу модель. Добавим больше эпох, увеличим размер хеша и подтянем лернинг рейт.

In [54]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 15.0 \
    --bit_precision 22 \
    --passes 100 \
    --cache -k

final_regressor = drugs.model.bin
Num weight bits = 22
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       30
256.536003 224.072006            2            2.0   1.0000  15.9690      145
328.243542 399.951080            4            4.0   7.0000  28.0000      157
1008.053105 1687.862669            8            8.0  24.0000  23.4970       73
3901.271561 6794.490017           16           16.0 224.0000   8.1367       38
2635.549805 1369.828049           32           32.0  25.0000  81.1440      147
2575.012004 2514.474203           64           64.0  13.0000  29.9650       77
1855.448868 1135.885732          128          128.0   6.0000   1.7608       80
1575.0869

In [55]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

only testing
predictions = drugs.preductions.txt
Num weight bits = 22
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000       74
544.334290 1052.668579            2            2.0  76.0000  43.5551       10
372.843262 201.352234            4            4.0   4.0000   0.0000       68
544.368635 715.894009            8            8.0  23.0000  48.4719       44
493.378039 442.387444           16           16.0   7.0000  21.1832      126
373.361964 253.345888           32           32.0  13.0000  21.3520       47
464.793796 556.225629           64           64.0   3.0000  10.0905      108
479.414651 494.035505          128          128.0   9.0000  41.7097      115
407.755910 336.097168          256          256.0  29.000

In [56]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.510615966968609

Так уже гораздо интереснее!

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_2.png" width="300">

#### Регуляризация

Можно попробовать добавить регуляризацию, однако нужно быть очень аккуратным с ней - она расчитывается для **каждого** объекта и поэтому ее нужно выставлять очень маленькой, чтобы модель не скатилась в 0.

In [57]:
%%time
# Попробуем использовать 0.00001 для L2, но скорее всего это будет очень много

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 15.0 \
    --bit_precision 22 \
    --passes 20 \
    --l2 0.00001 \
    --cache -k

using l2 regularization = 1e-05
final_regressor = drugs.model.bin
Num weight bits = 22
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       30
256.536034 224.072067            2            2.0   1.0000  15.9690      145
328.245617 399.955200            4            4.0   7.0000  28.0000      157
1008.064921 1687.884224            8            8.0  24.0000  23.4958       73
3901.193176 6794.321432           16           16.0 224.0000   8.1362       38
2635.421075 1369.648973           32           32.0  25.0000  81.1145      147
2574.827465 2514.233856           64           64.0  13.0000  29.9484       77
1854.434202 1134.040939          128          128.0   6.

In [58]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

only testing
predictions = drugs.preductions.txt
Num weight bits = 22
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
5.711040 5.711040            1            1.0   6.0000   3.6102       74
2302.700979 4599.690918            2            2.0  76.0000   8.1790       10
1197.536105 92.371231            4            4.0   4.0000   3.3018       68
946.584093 695.632082            8            8.0  23.0000  14.1478       44
508.381847 70.179600           16           16.0   7.0000  21.2303      126
559.358821 610.335794           32           32.0  13.0000  13.8739       47
645.041917 730.725014           64           64.0   3.0000  10.0210      108
1205.428500 1765.815082          128          128.0   9.0000  22.5388      115
1161.321458 1117.214417          256          256.0  29.0

In [59]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.0911722662960851

In [117]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 15.0 \
    --bit_precision 22 \
    --passes 100 \
    --cache -k

final_regressor = drugs.model.bin
Num weight bits = 22
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       30
256.536003 224.072006            2            2.0   1.0000  15.9690      145
328.243542 399.951080            4            4.0   7.0000  28.0000      157
1008.053105 1687.862669            8            8.0  24.0000  23.4970       73
3901.271561 6794.490017           16           16.0 224.0000   8.1367       38
2635.549805 1369.828049           32           32.0  25.0000  81.1440      147
2575.012004 2514.474203           64           64.0  13.0000  29.9650       77
1855.448868 1135.885732          128          128.0   6.0000   1.7608       80
1575.0869

In [118]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

only testing
predictions = drugs.preductions.txt
Num weight bits = 22
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000       74
663.856689 1291.713379            2            2.0  76.0000  40.0596       10
419.834221 175.811752            4            4.0   4.0000   0.0000       68
627.090374 834.346527            8            8.0  23.0000  48.1149       44
507.308460 387.526547           16           16.0   7.0000  25.1145      126
409.221345 311.134229           32           32.0  13.0000  19.5480       47
501.073717 592.926089           64           64.0   3.0000   7.7352      108
567.331315 633.588913          128          128.0   9.0000  33.9953      115
472.671125 378.010935          256          256.0  29.000

In [119]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')  # Все еще хуже

0.4738522304061916

Все еще хуже чем без регуляризации 

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_2.5.png" width="300">

Таким образом выставлять L* регуляризацию - это отдельное исскуство в VW. Можно попробовать выставлять сильно меньшее значение. Однако на моей практике очень редко получается руками подобрать хороший коэффициент - это возможно только если искать автоподбором. 

Так что совет - настраивайте L* регуляризации уже после настройки всех остальных параметров. Или вообще не используйте, VW имеет тенденцию неплохо обучаться и без нее.

#### Комбинируем признаки

Мы можем попробовать комбинировать (перемножать) признаки между собой, чтобы получать лучшее качество. Итоговое количество признаков сильно возрастет (если у нас есть признак с N значениями и с M значениями, то их перемножение даст еще N\*M признаков. Но нас это не смущает, так как у нас есть хешинг-трик! Можем себе позволить спокойно увеличивать количество признаков, не сильно переживая.

В нашем случае логично перемножить препараты и заболевания - признак препарат-заболевание звучит достаточно логично.

In [ ]:
template = "{target} |d {drug_name} |c {condition} |r {review} |s rating:{rating}"

In [ ]:
d r a g

In [63]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 15.0 \
    --bit_precision 22 \
    --passes 20 \
    --interactions dc \
    --cache -k

creating features for following interactions: dcr 
final_regressor = drugs.model.bin
Num weight bits = 22
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       56
173.775717 58.551434            2            2.0   1.0000   8.6519      286
236.528136 299.280556            4            4.0   7.0000  21.0339      310
1136.053465 2035.578795            8            8.0  24.0000  16.3336      142
3776.561829 6417.070192           16           16.0 224.0000   8.1378       72
2527.440120 1278.318411           32           32.0  25.0000  68.1992      290
2505.646199 2483.852278           64           64.0  13.0000  36.5816      150
1726.187350 946.728500          128   

In [64]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

creating features for following interactions: dcr 
only testing
predictions = drugs.preductions.txt
Num weight bits = 22
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000      144
85.240517 134.481033            2            2.0  76.0000  64.4034       16
615.387020 1145.533524            4            4.0   4.0000   3.1986      132
1816.075496 3016.763971            8            8.0  23.0000  23.8609       84
950.735275 85.395054           16           16.0   7.0000   6.4168      248
561.618642 172.502009           32           32.0  13.0000  13.1827       90
515.536312 469.453983           64           64.0   3.0000  13.6076      212
456.415955 397.295597          128          128.0   9.0000   9.6287      226
298.75

In [65]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.7498896088825375

Качество поднялось!

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_3.png" width="300">

Посчитаем количество признаков - 3431 + 884 + 49855 + 1 + 3431 \* 884 = **3 087 182**. Уже достаточно внушительное количество.

#### Используем N-граммы для текста

Можно попросить VW по определенному неймспейсу генерировать сразу N-граммы. Подход хорошо работает для текстов, так что мы можем попробовать использовать это для признака review.

Конечно этот еще сильнее взорвет количество признаков, но мы опять не сильно переживаем по этому поводу.

In [66]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 15.0 \
    --bit_precision 22 \
    --passes 20 \
    --ngram r2 \
    --cache -k

Generating 2-grams for r namespaces.
final_regressor = drugs.model.bin
Num weight bits = 22
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       55
173.416544 57.833088            2            2.0   1.0000   8.6048      285
234.932221 296.447899            4            4.0   7.0000  21.3651      309
1134.389025 2033.845829            8            8.0  24.0000  16.8272      141
3791.905764 6449.422503           16           16.0 224.0000   7.6771       71
2540.528538 1289.151311           32           32.0  25.0000  74.1889      289
2494.572742 2448.616947           64           64.0  13.0000  37.9520      149
1798.624631 1102.676520          128          128.0 

In [67]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

Generating 2-grams for r namespaces.
only testing
predictions = drugs.preductions.txt
Num weight bits = 22
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000      143
92.785980 149.571960            2            2.0  76.0000  63.7700       15
46.472919 0.159858            4            4.0   4.0000   3.7340      131
726.686506 1406.900093            8            8.0  23.0000  24.3720       83
467.138790 207.591073           16           16.0   7.0000   7.5358      247
583.989696 700.840603           32           32.0  13.0000  13.3271       89
293.150106 2.310516           64           64.0   3.0000   2.9172      211
240.087881 187.025655          128          128.0   9.0000  11.1393      225
197.618541 155.149201     

In [68]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.8619126818394689

Юху! Вот это уже реально похоже на какую-то модель, которую не стыдно и на продакшн катить!

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_4.png" width="300">

Количество признаков уже подобралось к отметке 3431 + 884 + 49855 + 1 + 49855 ^ 2 = **2 485 575 203**

#### Используем все вместе!

In [69]:
%%time

! vw --final_regressor drugs.model.bin drugs.train.vw \
    --learning_rate 20.0 \
    --bit_precision 23 \
    --passes 40 \
    --ngram r2 \
    --interactions dc \
    --cache -k

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
final_regressor = drugs.model.bin
Num weight bits = 23
learning rate = 20
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.train.vw.cache
Reading datafile = drugs.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
289.000000 289.000000            1            1.0  17.0000   0.0000       56
172.327087 55.654175            2            2.0   1.0000   8.4602      286
234.736885 297.146683            4            4.0   7.0000  21.2516      310
1134.530071 2034.323258            8            8.0  24.0000  17.2988      142
3821.997198 6509.464324           16           16.0 224.0000   7.8433       72
2605.829915 1389.662631           32           32.0  25.0000  82.3097      290
2535.426779 2465.023644           64           64.0  13.0000  39.4484      150
18

In [70]:
! vw --testonly --initial_regressor drugs.model.bin --predictions drugs.preductions.txt drugs.test.vw

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
only testing
predictions = drugs.preductions.txt
Num weight bits = 23
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
36.000000 36.000000            1            1.0   6.0000   0.0000      144
44.216021 52.432041            2            2.0  76.0000  68.7590       16
22.670023 1.124026            4            4.0   4.0000   3.6876      132
740.151412 1457.632801            8            8.0  23.0000  23.6109       84
453.024517 165.897623           16           16.0   7.0000   7.6341      248
663.569742 874.114967           32           32.0  13.0000  13.1812       90
332.543136 1.516530           64           64.0   3.0000   3.3430      212
275.126275 217.709414          128          128.0   9.

In [71]:
calc_r2('drugs.preductions.txt', 'drugs.test.vw')

0.8762065549926664

Лучший результат!

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_5.png" width="300">

Количество признаков перевалило за  3431 + 884 + 49855 + 1 + 7 + 49855 ^ 2 + 3431 \* 884=  **2 488 608 207**.

### Классификация

Помимо обычный линейрой регрессии с MSE , Vowpal Wabbit умеер решать и задачи классификации. Для этого нужно поменять обучаемую loss function. Список всех поддерживаемых loss можно найти здесь - https://github.com/VowpalWabbit/vowpal_wabbit/wiki/Loss-functions .

Для начала переформатируем наш датасет для классификации - сделаем два класса - полезный и не полезный отзыв. Определять это будем по количеству usefullCount - если больше медианы, то полезный, если меньше - нет.

In [72]:
df['usefulCount'].median()

16.0

Важно отметить, что для бинарной классификации, классы должны быть 1 и -1.

In [73]:
def convert_to_vw_class(data):
    target = 1 if data['usefulCount'] > 16.0 else -1
    
    drug_name = data['drugName'].lower().replace(' ', '_')
    condition = data['condition'].lower().replace(' ', '_')
    
    raw_text = data['review'].lower()
    word_pattern = re.compile(r"[a-zA-Z0-9_]+")
    words = [match.group(0) for match in re.finditer(word_pattern, raw_text)]
    review = ' '.join(words)
    
    rating = data['rating']

    
    template = "{target} |d {drug_name} |c {condition} |r {review} |s rating:{rating}"
    return template.format(
        target=target,
        drug_name=drug_name,
        condition=condition,
        review=review,
        rating=rating
    )

In [74]:
_, example = next(df.iterrows())
example

Unnamed: 0                                                206461
drugName                                               Valsartan
condition                           Left Ventricular Dysfunction
review         "It has no side effect, I take it in combinati...
rating                                                       9.0
date                                                May 20, 2012
usefulCount                                                   27
Name: 0, dtype: object

In [75]:
convert_to_vw_class(example)

'1 |d valsartan |c left_ventricular_dysfunction |r it has no side effect i take it in combination of bystolic 5 mg and fish oil |s rating:9.0'

In [76]:
def write_vw_class(dataframe, filename):
    with open(filename, 'w') as f:
        for vw_record in dataframe.parallel_apply(convert_to_vw_class, axis=1):
            f.write(vw_record + '\n')

In [77]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=420)

In [78]:
%%time

write_vw_class(df_train, 'drugs.class.train.vw')
write_vw_class(df_test, 'drugs.class.test.vw')

CPU times: user 853 ms, sys: 993 ms, total: 1.85 s
Wall time: 10.1 s


In [79]:
! head -n 3 drugs.class.train.vw

1 |d anaprox-ds |c pain |r i recommend if for girls if they have menstrual pain it helps alot 10 15 mins after i take the pill i feel no more pain |s rating:10.0
-1 |d levetiracetam |c epilepsy |r as a 42 year old female i have had seizures my entire life i have taken 14 anti convulsants at one point or another in an attempt to control my complex partial seizures but keppra is the worst medication i have ever taken the side effects are horrible although it 039 s the only medication that has been able to stop my seizures the side effects are so terrible that it 039 s not worth taking it i couldn 039 t string together a coherent thought had short term memory loss had awful mood swings and suddenly hated life everything was difficult i couldn 039 t think straight and felt emotionally raw all the time i could barely keep it together while on this medication every day of life was awful while on this medication i will never take it again |s rating:1.0
1 |d contrave |c obesity |r i have been 

In [80]:
%%time

! vw --final_regressor drugs.model.bin drugs.class.train.vw \
    --loss_function logistic \
    --learning_rate 20.0 \
    --bit_precision 23 \
    --passes 40 \
    --ngram r2 \
    --interactions dc \
    --cache -k

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
final_regressor = drugs.model.bin
Num weight bits = 23
learning rate = 20
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.class.train.vw.cache
Reading datafile = drugs.class.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
0.693147 0.693147            1            1.0   1.0000   0.0000       56
1.784606 2.876064            2            2.0  -1.0000   2.8181      286
1.551441 1.318277            4            4.0  -1.0000   1.9425      310
1.280395 1.009348            8            8.0   1.0000  -0.1680      142
1.133821 0.987248           16           16.0   1.0000  -0.6707       72
0.933702 0.733582           32           32.0   1.0000   3.8839      290
0.904595 0.875489           64           64.0  -1.0000   1.0534      150
0.844466 0.784337        

In [81]:
%%time

! vw drugs.class.test.vw \
    --testonly \
    --initial_regressor drugs.model.bin \
    --predictions predictions.txt

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
only testing
predictions = predictions.txt
Num weight bits = 23
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.class.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
45.949364 45.949364            1            1.0  -1.0000  -7.7786      144
28.838305 11.727246            2            2.0   1.0000   4.4245       16
36.769369 44.700434            4            4.0  -1.0000  -9.0952      132
28.369600 19.969830            8            8.0   1.0000   4.4692       84
25.758292 23.146984           16           16.0  -1.0000  -7.3193      248
26.630841 27.503391           32           32.0  -1.0000  -4.5997       90
41.285366 55.939890           64           64.0  -1.0000  -6.0912      212
38.983493 36.681621          128          128.0  -1.0000  -5

По умолчанию vw считает только регрессию - то есть значение логита для логистической регресии

In [82]:
! head predictions.txt

-7.778596
4.424507
5.885537
-9.095208
6.093756
1.256716
-7.066693
4.469213
-4.044488
6.824972


Чтобы предсказывать сразу классы для бинарной классификации можно указать флаг `--binary`

In [83]:
%%time

! vw drugs.class.test.vw \
    --binary \
    --testonly \
    --initial_regressor drugs.model.bin \
    --predictions predictions.txt

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
only testing
predictions = predictions.txt
Num weight bits = 23
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.class.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
0.000000 0.000000            1            1.0  -1.0000  -1.0000      144
0.000000 0.000000            2            2.0   1.0000   1.0000       16
0.000000 0.000000            4            4.0  -1.0000  -1.0000      132
0.125000 0.250000            8            8.0   1.0000   1.0000       84
0.125000 0.125000           16           16.0  -1.0000  -1.0000      248
0.093750 0.062500           32           32.0  -1.0000  -1.0000       90
0.046875 0.000000           64           64.0  -1.0000  -1.0000      212
0.039062 0.031250          128          128.0  -1.0000  -1.0000      226
0

In [84]:
! head predictions.txt

-1
1
1
-1
1
1
-1
1
-1
1


Чтобы предсказывать вероятности, то нужно указать vw что требуется пост-преобразование через `--link=logistic`

In [85]:
%%time

! vw drugs.class.test.vw \
    --link=logistic \
    --testonly \
    --initial_regressor drugs.model.bin \
    --predictions predictions.txt

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
only testing
predictions = predictions.txt
Num weight bits = 23
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.class.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
45.949364 45.949364            1            1.0  -1.0000   0.0004      144
28.838305 11.727246            2            2.0   1.0000   0.9882       16
36.769369 44.700434            4            4.0  -1.0000   0.0001      132
28.369600 19.969830            8            8.0   1.0000   0.9887       84
25.758292 23.146984           16           16.0  -1.0000   0.0007      248
26.630841 27.503391           32           32.0  -1.0000   0.0100       90
41.285366 55.939890           64           64.0  -1.0000   0.0023      212
38.983493 36.681621          128          128.0  -1.0000   0

In [86]:
! head predictions.txt

0.000418
0.988162
0.997228
0.000112
0.997748
0.778460
0.000852
0.988673
0.017217
0.998915


In [87]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score


def calc_class_metrics(predictions_filename, answers_filename):
    def read_target_from_vw(vw_record):
        return int((float(vw_record.split(' ')[0]) + 1) / 2)
    
    with open(predictions_filename, 'r') as f:
        y_pred = np.array([int(float(value) > 0.5) for value in f.readlines()])
        
    with open(answers_filename, 'r') as f:
        y_expected = np.array([read_target_from_vw(value) for value in f.readlines()])
        
    return accuracy_score(y_expected, y_pred), precision_score(y_expected, y_pred), recall_score(y_expected, y_pred)

In [88]:
calc_class_metrics('predictions.txt', 'drugs.class.test.vw')

(0.9601153366583541, 0.9561658387227794, 0.9636020151133501)

In [89]:
%%time

! vw --final_regressor drugs.model.bin drugs.class.train.vw \
    --loss_function logistic \
    --nn 3 \
    --inpass \
    --learning_rate 15.0 \
    --bit_precision 23 \
    --passes 50 \
    --ngram r2 \
    --interactions dc \
    --cache -k

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
final_regressor = drugs.model.bin
using input passthrough for neural network training
Num weight bits = 23
learning rate = 15
initial_t = 0
power_t = 0.5
decay_learning_rate = 1
creating cache_file = drugs.class.train.vw.cache
Reading datafile = drugs.class.train.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
0.693147 0.693147            1            1.0   1.0000   0.0000       56
1.941622 3.190097            2            2.0  -1.0000   3.1481      286
1.622740 1.303857            4            4.0  -1.0000   1.9788      310
1.305374 0.988007            8            8.0   1.0000  -0.0849      142
1.136837 0.968300           16           16.0   1.0000  -0.6021       72
0.934221 0.731605           32           32.0   1.0000   3.6791      290
0.903716 0.873211           64           64.0 

In [90]:
! vw --help

Num weight bits = 18
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = 
num sources = 1
driver:
  --onethread           Disable parse thread
VW options:
  --ring_size arg (=256, ) size of example ring
  --strict_parse           throw on malformed examples
Update options:
  -l [ --learning_rate ] arg Set learning rate
  --power_t arg              t power value
  --decay_learning_rate arg  Set Decay factor for learning_rate between passes
  --initial_t arg            initial t value
  --feature_mask arg         Use existing regressor to determine which 
                             parameters may be updated.  If no 
                             initial_regressor given, also used for initial 
                             weights.
Weight options:
  -i [ --initial_regressor ] arg  Initial regressor(s)
  --initial_weight arg            Set all weights to an initial value of arg.
  --random_weights                make initial weights random
  --normal_weights  

In [91]:
%%time

! vw drugs.class.test.vw \
    --link=logistic \
    --testonly \
    --initial_regressor drugs.model.bin \
    --predictions predictions.txt

Generating 2-grams for r namespaces.
creating features for following interactions: dc 
only testing
predictions = predictions.txt
using input passthrough for neural network testing
Num weight bits = 23
learning rate = 0.5
initial_t = 0
power_t = 0.5
using no cache
Reading datafile = drugs.class.test.vw
num sources = 1
average  since         example        example  current  current  current
loss     last          counter         weight    label  predict features
50.758190 50.758190            1            1.0  -1.0000   0.0003      144
30.176360 9.594529            2            2.0   1.0000   0.9837       16
35.817200 41.458041            4            4.0  -1.0000   0.0002      132
26.169814 16.522428            8            8.0   1.0000   0.9843       84
23.266625 20.363437           16           16.0  -1.0000   0.0008      248
24.219722 25.172818           32           32.0  -1.0000   0.0252       90
36.875913 49.532104           64           64.0  -1.0000   0.0060      212
33.922824 

In [92]:
calc_class_metrics('predictions.txt', 'drugs.class.test.vw')

(0.9593827930174563, 0.953857649915935, 0.9646095717884131)

### Нативная работа с категориальными признаками. Деревья решений.

Как можно было видеть выше, огромное количество признаков может возникать из-за кодирования больших категориальных признаков или текста. Есть алтернативный способ работы с такими признаками - использование деревьев решений. Эти модели могут использовать изначальные признаки "как есть", а значит и нет вообще пробелмы с количеством признаков.

Один из хороших примерно инструментов, которые активно используют возможности деревьев для работы с категориями - CatBoost. Сама приставка Cat = Categorical, показывая что эта версия бустинга деревьев очень хорошо справляется с таким типом данных.

In [121]:
pip list | grep "catboost"

catboost                  1.2.10
Note: you may need to restart the kernel to use updated packages.


In [93]:
from catboost import Pool, CatBoostClassifier, CatBoostRegressor

In [94]:
df

,Unnamed: 0,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9.0,"May 20, 2012",27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17
3,138000,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10
4,35696,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9.0,"November 27, 2016",37
...,...,...,...,...,...,...,...
161292,191035,Campral,Alcohol Dependence,"""I wrote my first report in Mid-October of 201...",10.0,"May 31, 2015",125
161293,127085,Metoclopramide,Nausea/Vomiting,"""I was given this in IV before surgey. I immed...",1.0,"November 1, 2011",34
161294,187382,Orencia,Rheumatoid Arthritis,"""Limited improvement after 4 months, developed...",2.0,"March 15, 2014",35
161295,47128,Thyroid desiccated,Underactive Thyroid,"""I&#039;ve been on thyroid medication 49 years...",10.0,"September 19, 2015",79


**Историческая заметка:**
Долгое время не было возможности использовать текстовые признаки в задаче регрессии. И каждый предыдущий год я просил студентов набегать на вот этот тикет и ставить лайки задаче - https://github.com/catboost/catboost/issues/1221 .

И вот, свершилось! 

In [95]:
X = df[['drugName', 'condition', 'review', 'rating']]
Y = df['usefulCount']

In [96]:
Y

0          27
1         192
2          17
3          10
4          37
         ... 
161292    125
161293     34
161294     35
161295     79
161296    116
Name: usefulCount, Length: 320796, dtype: int64

In [97]:
X

,drugName,condition,review,rating
0,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9.0
1,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8.0
2,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5.0
3,Ortho Evra,Birth Control,"""This is my first time using any form of birth...",8.0
4,Buprenorphine / naloxone,Opiate Dependence,"""Suboxone has completely turned my life around...",9.0
...,...,...,...,...
161292,Campral,Alcohol Dependence,"""I wrote my first report in Mid-October of 201...",10.0
161293,Metoclopramide,Nausea/Vomiting,"""I was given this in IV before surgey. I immed...",1.0
161294,Orencia,Rheumatoid Arthritis,"""Limited improvement after 4 months, developed...",2.0
161295,Thyroid desiccated,Underactive Thyroid,"""I&#039;ve been on thyroid medication 49 years...",10.0


In [98]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=420)

In [99]:
train_pool = Pool(
    X_train, 
    y_train, 
    cat_features=[0, 1], 
    text_features=[2], 
    feature_names=list(X_train.columns)
)
test_pool = Pool(
    X_test, 
    cat_features=[0, 1], 
    text_features=[2], 
    feature_names=list(X_test.columns)
)

In [100]:
model = CatBoostRegressor()

In [101]:
%%time

model.fit(train_pool, verbose=True, plot=False)

Learning rate set to 0.098368
0:	learn: 35.2890663	total: 491ms	remaining: 8m 10s
1:	learn: 34.6127370	total: 976ms	remaining: 8m 6s
2:	learn: 34.0495009	total: 1.41s	remaining: 7m 47s
3:	learn: 33.5874955	total: 1.87s	remaining: 7m 45s
4:	learn: 33.2003497	total: 2.31s	remaining: 7m 38s
5:	learn: 32.8785335	total: 2.77s	remaining: 7m 38s
6:	learn: 32.4720987	total: 3.19s	remaining: 7m 31s
7:	learn: 32.1373289	total: 3.57s	remaining: 7m 23s
8:	learn: 31.8574671	total: 3.99s	remaining: 7m 19s
9:	learn: 31.6188589	total: 4.36s	remaining: 7m 11s
10:	learn: 31.4134171	total: 4.73s	remaining: 7m 5s
11:	learn: 31.2420471	total: 5.12s	remaining: 7m 1s
12:	learn: 31.0951054	total: 5.55s	remaining: 7m 1s
13:	learn: 30.9746579	total: 5.92s	remaining: 6m 56s
14:	learn: 30.8686787	total: 6.28s	remaining: 6m 52s
15:	learn: 30.7795885	total: 6.62s	remaining: 6m 47s
16:	learn: 30.7046092	total: 7.01s	remaining: 6m 45s
17:	learn: 30.6369182	total: 7.38s	remaining: 6m 42s
18:	learn: 30.5632258	total: 7

155:	learn: 28.5123044	total: 45.6s	remaining: 4m 6s
156:	learn: 28.5052673	total: 45.9s	remaining: 4m 6s
157:	learn: 28.4965354	total: 46.1s	remaining: 4m 5s
158:	learn: 28.4880542	total: 46.4s	remaining: 4m 5s
159:	learn: 28.4802618	total: 46.7s	remaining: 4m 4s
160:	learn: 28.4698060	total: 46.9s	remaining: 4m 4s
161:	learn: 28.4612163	total: 47.2s	remaining: 4m 4s
162:	learn: 28.4553066	total: 47.5s	remaining: 4m 3s
163:	learn: 28.4453321	total: 47.7s	remaining: 4m 3s
164:	learn: 28.4362955	total: 48s	remaining: 4m 2s
165:	learn: 28.4293722	total: 48.3s	remaining: 4m 2s
166:	learn: 28.4188359	total: 48.7s	remaining: 4m 2s
167:	learn: 28.4071181	total: 49s	remaining: 4m 2s
168:	learn: 28.3968046	total: 49.3s	remaining: 4m 2s
169:	learn: 28.3845246	total: 49.5s	remaining: 4m 1s
170:	learn: 28.3734754	total: 49.8s	remaining: 4m 1s
171:	learn: 28.3670468	total: 50s	remaining: 4m
172:	learn: 28.3608382	total: 50.3s	remaining: 4m
173:	learn: 28.3527426	total: 50.5s	remaining: 3m 59s
174:

307:	learn: 27.3991939	total: 1m 25s	remaining: 3m 12s
308:	learn: 27.3952327	total: 1m 25s	remaining: 3m 12s
309:	learn: 27.3901810	total: 1m 26s	remaining: 3m 11s
310:	learn: 27.3850911	total: 1m 26s	remaining: 3m 11s
311:	learn: 27.3802108	total: 1m 26s	remaining: 3m 11s
312:	learn: 27.3718849	total: 1m 27s	remaining: 3m 11s
313:	learn: 27.3652155	total: 1m 27s	remaining: 3m 10s
314:	learn: 27.3596295	total: 1m 27s	remaining: 3m 10s
315:	learn: 27.3522613	total: 1m 27s	remaining: 3m 10s
316:	learn: 27.3467594	total: 1m 28s	remaining: 3m 9s
317:	learn: 27.3401480	total: 1m 28s	remaining: 3m 9s
318:	learn: 27.3343013	total: 1m 28s	remaining: 3m 9s
319:	learn: 27.3300322	total: 1m 28s	remaining: 3m 8s
320:	learn: 27.3235162	total: 1m 29s	remaining: 3m 8s
321:	learn: 27.3162951	total: 1m 29s	remaining: 3m 8s
322:	learn: 27.3123793	total: 1m 29s	remaining: 3m 7s
323:	learn: 27.3040742	total: 1m 29s	remaining: 3m 7s
324:	learn: 27.2968658	total: 1m 30s	remaining: 3m 7s
325:	learn: 27.2924

458:	learn: 26.5666652	total: 2m 4s	remaining: 2m 26s
459:	learn: 26.5590295	total: 2m 4s	remaining: 2m 26s
460:	learn: 26.5531843	total: 2m 5s	remaining: 2m 26s
461:	learn: 26.5468556	total: 2m 5s	remaining: 2m 26s
462:	learn: 26.5403104	total: 2m 5s	remaining: 2m 25s
463:	learn: 26.5353765	total: 2m 6s	remaining: 2m 25s
464:	learn: 26.5308581	total: 2m 6s	remaining: 2m 25s
465:	learn: 26.5257643	total: 2m 6s	remaining: 2m 24s
466:	learn: 26.5204138	total: 2m 6s	remaining: 2m 24s
467:	learn: 26.5161411	total: 2m 7s	remaining: 2m 24s
468:	learn: 26.5101275	total: 2m 7s	remaining: 2m 24s
469:	learn: 26.5038269	total: 2m 7s	remaining: 2m 23s
470:	learn: 26.4989704	total: 2m 7s	remaining: 2m 23s
471:	learn: 26.4926857	total: 2m 8s	remaining: 2m 23s
472:	learn: 26.4879088	total: 2m 8s	remaining: 2m 23s
473:	learn: 26.4849966	total: 2m 8s	remaining: 2m 22s
474:	learn: 26.4806746	total: 2m 8s	remaining: 2m 22s
475:	learn: 26.4738708	total: 2m 9s	remaining: 2m 22s
476:	learn: 26.4690172	total

609:	learn: 25.8512586	total: 2m 43s	remaining: 1m 44s
610:	learn: 25.8478308	total: 2m 44s	remaining: 1m 44s
611:	learn: 25.8439592	total: 2m 44s	remaining: 1m 44s
612:	learn: 25.8391093	total: 2m 44s	remaining: 1m 43s
613:	learn: 25.8355799	total: 2m 44s	remaining: 1m 43s
614:	learn: 25.8290068	total: 2m 45s	remaining: 1m 43s
615:	learn: 25.8238277	total: 2m 45s	remaining: 1m 43s
616:	learn: 25.8212702	total: 2m 45s	remaining: 1m 42s
617:	learn: 25.8168361	total: 2m 45s	remaining: 1m 42s
618:	learn: 25.8117160	total: 2m 46s	remaining: 1m 42s
619:	learn: 25.8081325	total: 2m 46s	remaining: 1m 41s
620:	learn: 25.8045045	total: 2m 46s	remaining: 1m 41s
621:	learn: 25.7975089	total: 2m 46s	remaining: 1m 41s
622:	learn: 25.7937496	total: 2m 47s	remaining: 1m 41s
623:	learn: 25.7894573	total: 2m 47s	remaining: 1m 40s
624:	learn: 25.7865072	total: 2m 47s	remaining: 1m 40s
625:	learn: 25.7832759	total: 2m 47s	remaining: 1m 40s
626:	learn: 25.7774577	total: 2m 48s	remaining: 1m 40s
627:	learn

760:	learn: 25.2491459	total: 3m 22s	remaining: 1m 3s
761:	learn: 25.2447874	total: 3m 22s	remaining: 1m 3s
762:	learn: 25.2412576	total: 3m 22s	remaining: 1m 3s
763:	learn: 25.2376743	total: 3m 23s	remaining: 1m 2s
764:	learn: 25.2351323	total: 3m 23s	remaining: 1m 2s
765:	learn: 25.2319512	total: 3m 23s	remaining: 1m 2s
766:	learn: 25.2273097	total: 3m 23s	remaining: 1m 1s
767:	learn: 25.2231289	total: 3m 24s	remaining: 1m 1s
768:	learn: 25.2189838	total: 3m 24s	remaining: 1m 1s
769:	learn: 25.2143238	total: 3m 24s	remaining: 1m 1s
770:	learn: 25.2103910	total: 3m 24s	remaining: 1m
771:	learn: 25.2078978	total: 3m 25s	remaining: 1m
772:	learn: 25.2035597	total: 3m 25s	remaining: 1m
773:	learn: 25.1995308	total: 3m 25s	remaining: 1m
774:	learn: 25.1932646	total: 3m 26s	remaining: 59.8s
775:	learn: 25.1881042	total: 3m 26s	remaining: 59.5s
776:	learn: 25.1845538	total: 3m 26s	remaining: 59.3s
777:	learn: 25.1807604	total: 3m 26s	remaining: 59s
778:	learn: 25.1765796	total: 3m 27s	remai

913:	learn: 24.6950172	total: 4m 1s	remaining: 22.7s
914:	learn: 24.6923687	total: 4m 1s	remaining: 22.5s
915:	learn: 24.6896539	total: 4m 2s	remaining: 22.2s
916:	learn: 24.6861009	total: 4m 2s	remaining: 21.9s
917:	learn: 24.6830322	total: 4m 2s	remaining: 21.7s
918:	learn: 24.6801329	total: 4m 2s	remaining: 21.4s
919:	learn: 24.6774057	total: 4m 3s	remaining: 21.1s
920:	learn: 24.6749858	total: 4m 3s	remaining: 20.9s
921:	learn: 24.6728378	total: 4m 3s	remaining: 20.6s
922:	learn: 24.6689094	total: 4m 3s	remaining: 20.4s
923:	learn: 24.6662837	total: 4m 4s	remaining: 20.1s
924:	learn: 24.6608547	total: 4m 4s	remaining: 19.8s
925:	learn: 24.6566507	total: 4m 4s	remaining: 19.6s
926:	learn: 24.6533424	total: 4m 5s	remaining: 19.3s
927:	learn: 24.6502775	total: 4m 5s	remaining: 19s
928:	learn: 24.6462802	total: 4m 5s	remaining: 18.8s
929:	learn: 24.6434063	total: 4m 5s	remaining: 18.5s
930:	learn: 24.6395670	total: 4m 6s	remaining: 18.2s
931:	learn: 24.6360728	total: 4m 6s	remaining: 1

CatBoostRegressor(loss_function='RMSE')

In [102]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [103]:
y_pred = model.predict(test_pool)
print(r2_score(y_test, y_pred))
print(mean_squared_error(y_test, y_pred))

0.5302906112959198
676.7751911342218


In [104]:
model.plot_tree(
    tree_idx=0,
    pool=train_pool
)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

Попробуем теперь тоже самое, но для классификации

In [105]:
df['usefulCount'].median()

16.0

In [106]:
X = df[['drugName', 'condition', 'review', 'rating']]
Y = df['usefulCount'].apply(lambda x: 1 if x > 16.0 else 0)

In [107]:
Y

0         1
1         1
2         1
3         0
4         1
         ..
161292    1
161293    1
161294    1
161295    1
161296    1
Name: usefulCount, Length: 320796, dtype: int64

In [108]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=420)

In [109]:
train_pool = Pool(
    X_train, 
    y_train, 
    cat_features=[0, 1], 
    text_features=[2], 
    feature_names=list(X_train.columns)
)
test_pool = Pool(
    X_test, 
    cat_features=[0, 1], 
    text_features=[2], 
    feature_names=list(X_test.columns)
)

In [110]:
model = CatBoostClassifier()

In [111]:
%%time

model.fit(train_pool, verbose=True, plot=False)

Learning rate set to 0.110081
0:	learn: 0.6547231	total: 468ms	remaining: 7m 47s
1:	learn: 0.6254482	total: 940ms	remaining: 7m 49s
2:	learn: 0.6027517	total: 1.4s	remaining: 7m 44s
3:	learn: 0.5868032	total: 1.84s	remaining: 7m 39s
4:	learn: 0.5727374	total: 2.27s	remaining: 7m 32s
5:	learn: 0.5616695	total: 2.71s	remaining: 7m 29s
6:	learn: 0.5475251	total: 3.21s	remaining: 7m 34s
7:	learn: 0.5361624	total: 3.67s	remaining: 7m 35s
8:	learn: 0.5278337	total: 4.13s	remaining: 7m 34s
9:	learn: 0.5233409	total: 4.5s	remaining: 7m 25s
10:	learn: 0.5171006	total: 4.98s	remaining: 7m 27s
11:	learn: 0.5117420	total: 5.43s	remaining: 7m 26s
12:	learn: 0.5083562	total: 5.81s	remaining: 7m 21s
13:	learn: 0.5049678	total: 6.31s	remaining: 7m 24s
14:	learn: 0.5027707	total: 6.7s	remaining: 7m 20s
15:	learn: 0.5002646	total: 7.15s	remaining: 7m 19s
16:	learn: 0.4985155	total: 7.53s	remaining: 7m 15s
17:	learn: 0.4967042	total: 7.92s	remaining: 7m 12s
18:	learn: 0.4952188	total: 8.34s	remaining: 7m

158:	learn: 0.4615675	total: 56.8s	remaining: 5m
159:	learn: 0.4613856	total: 57.3s	remaining: 5m
160:	learn: 0.4612566	total: 57.6s	remaining: 5m
161:	learn: 0.4611128	total: 57.9s	remaining: 4m 59s
162:	learn: 0.4610096	total: 58.2s	remaining: 4m 58s
163:	learn: 0.4608994	total: 58.5s	remaining: 4m 58s
164:	learn: 0.4607604	total: 58.8s	remaining: 4m 57s
165:	learn: 0.4606428	total: 59.1s	remaining: 4m 57s
166:	learn: 0.4605811	total: 59.5s	remaining: 4m 56s
167:	learn: 0.4604403	total: 59.8s	remaining: 4m 56s
168:	learn: 0.4603740	total: 1m	remaining: 4m 55s
169:	learn: 0.4602557	total: 1m	remaining: 4m 55s
170:	learn: 0.4601148	total: 1m	remaining: 4m 54s
171:	learn: 0.4599619	total: 1m 1s	remaining: 4m 53s
172:	learn: 0.4598885	total: 1m 1s	remaining: 4m 53s
173:	learn: 0.4597589	total: 1m 1s	remaining: 4m 52s
174:	learn: 0.4596285	total: 1m 1s	remaining: 4m 52s
175:	learn: 0.4595037	total: 1m 2s	remaining: 4m 51s
176:	learn: 0.4593644	total: 1m 2s	remaining: 4m 51s
177:	learn: 0.

312:	learn: 0.4440674	total: 1m 45s	remaining: 3m 52s
313:	learn: 0.4439545	total: 1m 46s	remaining: 3m 52s
314:	learn: 0.4438948	total: 1m 46s	remaining: 3m 51s
315:	learn: 0.4437617	total: 1m 46s	remaining: 3m 51s
316:	learn: 0.4436395	total: 1m 47s	remaining: 3m 51s
317:	learn: 0.4435601	total: 1m 47s	remaining: 3m 50s
318:	learn: 0.4434516	total: 1m 47s	remaining: 3m 50s
319:	learn: 0.4433612	total: 1m 48s	remaining: 3m 49s
320:	learn: 0.4432586	total: 1m 48s	remaining: 3m 49s
321:	learn: 0.4431399	total: 1m 48s	remaining: 3m 48s
322:	learn: 0.4430087	total: 1m 49s	remaining: 3m 48s
323:	learn: 0.4428939	total: 1m 49s	remaining: 3m 48s
324:	learn: 0.4428049	total: 1m 49s	remaining: 3m 47s
325:	learn: 0.4427138	total: 1m 50s	remaining: 3m 47s
326:	learn: 0.4425883	total: 1m 50s	remaining: 3m 47s
327:	learn: 0.4424702	total: 1m 50s	remaining: 3m 46s
328:	learn: 0.4423729	total: 1m 51s	remaining: 3m 46s
329:	learn: 0.4422588	total: 1m 51s	remaining: 3m 46s
330:	learn: 0.4421503	total:

466:	learn: 0.4290491	total: 2m 33s	remaining: 2m 55s
467:	learn: 0.4289873	total: 2m 34s	remaining: 2m 55s
468:	learn: 0.4288860	total: 2m 34s	remaining: 2m 54s
469:	learn: 0.4288076	total: 2m 34s	remaining: 2m 54s
470:	learn: 0.4286969	total: 2m 34s	remaining: 2m 54s
471:	learn: 0.4285994	total: 2m 35s	remaining: 2m 53s
472:	learn: 0.4285100	total: 2m 35s	remaining: 2m 53s
473:	learn: 0.4284409	total: 2m 35s	remaining: 2m 52s
474:	learn: 0.4283483	total: 2m 36s	remaining: 2m 52s
475:	learn: 0.4282510	total: 2m 36s	remaining: 2m 52s
476:	learn: 0.4281711	total: 2m 36s	remaining: 2m 51s
477:	learn: 0.4280916	total: 2m 37s	remaining: 2m 51s
478:	learn: 0.4279997	total: 2m 37s	remaining: 2m 51s
479:	learn: 0.4279288	total: 2m 37s	remaining: 2m 50s
480:	learn: 0.4278324	total: 2m 37s	remaining: 2m 50s
481:	learn: 0.4277458	total: 2m 38s	remaining: 2m 50s
482:	learn: 0.4276508	total: 2m 38s	remaining: 2m 49s
483:	learn: 0.4275858	total: 2m 38s	remaining: 2m 49s
484:	learn: 0.4274901	total:

619:	learn: 0.4159509	total: 3m 21s	remaining: 2m 3s
620:	learn: 0.4158698	total: 3m 21s	remaining: 2m 2s
621:	learn: 0.4157549	total: 3m 21s	remaining: 2m 2s
622:	learn: 0.4156821	total: 3m 22s	remaining: 2m 2s
623:	learn: 0.4155989	total: 3m 22s	remaining: 2m 1s
624:	learn: 0.4155250	total: 3m 22s	remaining: 2m 1s
625:	learn: 0.4154782	total: 3m 22s	remaining: 2m 1s
626:	learn: 0.4153948	total: 3m 23s	remaining: 2m
627:	learn: 0.4153069	total: 3m 23s	remaining: 2m
628:	learn: 0.4152554	total: 3m 23s	remaining: 2m
629:	learn: 0.4151750	total: 3m 24s	remaining: 1m 59s
630:	learn: 0.4151076	total: 3m 24s	remaining: 1m 59s
631:	learn: 0.4150325	total: 3m 24s	remaining: 1m 59s
632:	learn: 0.4149552	total: 3m 25s	remaining: 1m 58s
633:	learn: 0.4148180	total: 3m 25s	remaining: 1m 58s
634:	learn: 0.4147389	total: 3m 25s	remaining: 1m 58s
635:	learn: 0.4146510	total: 3m 26s	remaining: 1m 57s
636:	learn: 0.4145725	total: 3m 26s	remaining: 1m 57s
637:	learn: 0.4144824	total: 3m 26s	remaining: 

772:	learn: 0.4036083	total: 4m 8s	remaining: 1m 13s
773:	learn: 0.4035112	total: 4m 8s	remaining: 1m 12s
774:	learn: 0.4034165	total: 4m 9s	remaining: 1m 12s
775:	learn: 0.4033320	total: 4m 9s	remaining: 1m 12s
776:	learn: 0.4032697	total: 4m 9s	remaining: 1m 11s
777:	learn: 0.4032019	total: 4m 10s	remaining: 1m 11s
778:	learn: 0.4031279	total: 4m 10s	remaining: 1m 11s
779:	learn: 0.4030489	total: 4m 10s	remaining: 1m 10s
780:	learn: 0.4029773	total: 4m 11s	remaining: 1m 10s
781:	learn: 0.4028790	total: 4m 11s	remaining: 1m 10s
782:	learn: 0.4028207	total: 4m 11s	remaining: 1m 9s
783:	learn: 0.4027535	total: 4m 11s	remaining: 1m 9s
784:	learn: 0.4027004	total: 4m 12s	remaining: 1m 9s
785:	learn: 0.4026080	total: 4m 12s	remaining: 1m 8s
786:	learn: 0.4025004	total: 4m 12s	remaining: 1m 8s
787:	learn: 0.4024422	total: 4m 13s	remaining: 1m 8s
788:	learn: 0.4023594	total: 4m 13s	remaining: 1m 7s
789:	learn: 0.4022897	total: 4m 13s	remaining: 1m 7s
790:	learn: 0.4021987	total: 4m 14s	remai

928:	learn: 0.3920390	total: 4m 56s	remaining: 22.6s
929:	learn: 0.3919582	total: 4m 56s	remaining: 22.3s
930:	learn: 0.3918962	total: 4m 56s	remaining: 22s
931:	learn: 0.3918235	total: 4m 57s	remaining: 21.7s
932:	learn: 0.3917480	total: 4m 57s	remaining: 21.3s
933:	learn: 0.3916649	total: 4m 57s	remaining: 21s
934:	learn: 0.3915789	total: 4m 57s	remaining: 20.7s
935:	learn: 0.3915132	total: 4m 58s	remaining: 20.4s
936:	learn: 0.3914698	total: 4m 58s	remaining: 20.1s
937:	learn: 0.3914031	total: 4m 58s	remaining: 19.7s
938:	learn: 0.3912966	total: 4m 59s	remaining: 19.4s
939:	learn: 0.3912327	total: 4m 59s	remaining: 19.1s
940:	learn: 0.3911800	total: 4m 59s	remaining: 18.8s
941:	learn: 0.3911405	total: 5m	remaining: 18.5s
942:	learn: 0.3910655	total: 5m	remaining: 18.2s
943:	learn: 0.3910011	total: 5m	remaining: 17.8s
944:	learn: 0.3909289	total: 5m 1s	remaining: 17.5s
945:	learn: 0.3908439	total: 5m 1s	remaining: 17.2s
946:	learn: 0.3907894	total: 5m 1s	remaining: 16.9s
947:	learn: 

CatBoostClassifier()

In [112]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

In [113]:
y_pred = model.predict(test_pool)
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred))
print(recall_score(y_test, y_pred))

0.8188279301745636
0.7983641536273115
0.8482367758186398


In [114]:
model.plot_tree(
    tree_idx=0,
    pool=train_pool
)

ExecutableNotFound: failed to execute PosixPath('dot'), make sure the Graphviz executables are on your systems' PATH

Этот результат хорош сразу по многим параметрам

* Само значение качества достаточно высокое с первого запуска.
* Мы вообще практически ничего не делали с данными. Только дату немного отформатировали и сразу кинули в модель.
* Мы ничего не делали с моделью. Никаких дополнительных параметров настраивать не пришлось.
* Количество признаков у модели - всего 5! Нам не пришлось делать лишние признаки чтобы подать их в обучение.

<img src="https://raw.githubusercontent.com/ADKosm/lsml-2021-public/main/imgs/vince_mcmahon_6.png" width="300">